# 00 — Verify Connection

Confirms three things before the demo can run:

1. **Spark / Databricks Connect** session is live (or you're in the Databricks workspace where `spark` is already there)
2. **Unity Catalog target schemas** exist — creates `{UC_SCHEMA}_bronze`, `_silver`, `_gold` if needed
3. **SportLogiq API** credentials work — performs a single login probe and lists competitions

Run this first. Every other notebook in the demo expects the schemas and the Volume to exist.

In [ ]:
import os, json
from dotenv import load_dotenv
load_dotenv()

# Dual-mode Spark session — works locally via Databricks Connect AND inside a
# Databricks workspace notebook. In the workspace, `spark` already exists.
try:
    spark
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().getOrCreate()

from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

In [ ]:
UC_CATALOG    = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA     = os.getenv("UC_SCHEMA", "sportlogiq_nhl")
BRONZE_SCHEMA = f"{UC_SCHEMA}_bronze"
SILVER_SCHEMA = f"{UC_SCHEMA}_silver"
GOLD_SCHEMA   = f"{UC_SCHEMA}_gold"
VOLUME_NAME   = "raw_data"
VOLUME_PATH   = f"/Volumes/{UC_CATALOG}/{BRONZE_SCHEMA}/{VOLUME_NAME}"

print(f"Catalog: {UC_CATALOG}")
print(f"Bronze:  {UC_CATALOG}.{BRONZE_SCHEMA}")
print(f"Silver:  {UC_CATALOG}.{SILVER_SCHEMA}")
print(f"Gold:    {UC_CATALOG}.{GOLD_SCHEMA}")
print(f"Volume:  {VOLUME_PATH}")

In [ ]:
# Spark + workspace identity
spark.sql("SELECT current_user(), current_timestamp(), current_catalog()").show(truncate=False)
print("Spark version:", spark.version)

In [ ]:
# Create the three medallion schemas + the raw_data Volume on bronze
for schema in [BRONZE_SCHEMA, SILVER_SCHEMA, GOLD_SCHEMA]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {UC_CATALOG}.{schema}")
    print(f"  schema ready: {UC_CATALOG}.{schema}")

spark.sql(f"CREATE VOLUME IF NOT EXISTS {UC_CATALOG}.{BRONZE_SCHEMA}.{VOLUME_NAME}")
print(f"  volume ready: {VOLUME_PATH}")

## SportLogiq API probe

Logs in once and lists competitions to confirm credentials are good. If you
see a 401 / `SportLogiqAPIError` here, fix the creds before moving on — every
later notebook depends on this client working.

In [ ]:
def get_secret(env_var: str, scope: str = "sportlogiq", key: str | None = None) -> str:
    """Read a secret from .env first, then fall back to a Databricks secret scope.

    Lets the same notebook run from a laptop (.env) or inside a Databricks
    Git Folder (workspace secret scope `sportlogiq`).
    """
    val = os.getenv(env_var)
    if val:
        return val
    try:
        from pyspark.dbutils import DBUtils  # type: ignore
        dbutils = DBUtils(spark)
        return dbutils.secrets.get(scope=scope, key=key or env_var.lower())
    except Exception as e:
        raise RuntimeError(
            f"Could not resolve {env_var}: not in environment and Databricks "
            f"secret scope '{scope}' lookup failed ({e})."
        )

In [ ]:
from sportlogiq_client import SportLogiqAPI

api = SportLogiqAPI(
    username=get_secret("SPORTLOGIQ_USERNAME"),
    password=get_secret("SPORTLOGIQ_PASSWORD"),
)
print("Login OK")

competitions = api.get_competitions().json()
print(f"Competitions visible: {len(competitions.get('competitions', []))}")
for c in competitions.get("competitions", [])[:5]:
    print(f"  - {c.get('id')}: {c.get('name')}")

## Optional reset

Uncomment to wipe the demo schemas (preserves the Volume so you don't have to re-pull data).

In [ ]:
# for schema in [BRONZE_SCHEMA, SILVER_SCHEMA, GOLD_SCHEMA]:
#     spark.sql(f"DROP SCHEMA IF EXISTS {UC_CATALOG}.{schema} CASCADE")
#     spark.sql(f"CREATE SCHEMA IF NOT EXISTS {UC_CATALOG}.{schema}")
# print("Schemas reset.")